In [ ]:
# Use resnet20_quant model

# Do not quantize the first conv layer or the last FC layer

# Train the 4-bit model to 85-90% accuracy

# 2nd conv layer: Find integer inputs x_int and integer weight w_int 
# Similar to [code5] vgg16 example

# Compute Partial sum from x_int and w_int
# Create nn.conv2d(16, 16, kernel_size=3, stride=1, padding=1, bias=False)
# Set weight to w_in
# Then rescale the integer psum with the quantization step sizes
# Compare with original output of that layer (forward prehook)

# Pretty much the same as [Code7], just using resnet20_quant model 

import argparse 
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

from models import *

global best_prec

use_gpu = torch.cuda.is_available()

print('=> Building model...')
    
    
batch_size = 128

model_name = "cifar10"
model = resnet20_quant(float=False, bits=2) # Use float=False for quantized version

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.reshape(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res

class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1     


=> Building model...
Files already downloaded and verified
Files already downloaded and verified


In [18]:
# Train the 4-bit model to 85-90% accuracy
# Note: QuantConv2d class is already 4-bit by default

model = resnet20_quant(float=False)

# This cell is from the website

lr = 0.1

weight_decay = 5e-4
epochs = 60
best_prec = 0

model = model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 45], gamma=0.1)

# weight decay: for regularization to prevent overfitting


if not os.path.exists('result'):
    os.makedirs('result')
    
fdir = 'result/'+str(model_name)

if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    scheduler.step()

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)
    

Epoch: [0][0/391]	Time 43.099 (43.099)	Data 42.596 (42.596)	Loss 2.3814 (2.3814)	Prec 6.250% (6.250%)
Epoch: [0][100/391]	Time 0.078 (0.510)	Data 0.002 (0.423)	Loss 1.7857 (1.9590)	Prec 28.906% (26.153%)
Epoch: [0][200/391]	Time 0.094 (0.297)	Data 0.002 (0.213)	Loss 1.5106 (1.8112)	Prec 41.406% (31.495%)
Epoch: [0][300/391]	Time 0.080 (0.225)	Data 0.002 (0.143)	Loss 1.3657 (1.6979)	Prec 50.000% (36.124%)
Validation starts
Test: [0/79]	Time 17.263 (17.263)	Loss 1.4800 (1.4800)	Prec 51.562% (51.562%)
 * Prec 46.660% 
best acc: 46.660000
Epoch: [1][0/391]	Time 16.176 (16.176)	Data 15.914 (15.914)	Loss 1.3209 (1.3209)	Prec 54.688% (54.688%)
Epoch: [1][100/391]	Time 0.078 (0.286)	Data 0.002 (0.159)	Loss 1.0959 (1.2157)	Prec 60.938% (56.072%)
Epoch: [1][200/391]	Time 0.077 (0.184)	Data 0.002 (0.081)	Loss 1.1619 (1.1850)	Prec 60.156% (57.175%)
Epoch: [1][300/391]	Time 0.080 (0.150)	Data 0.002 (0.054)	Loss 1.0503 (1.1490)	Prec 59.375% (58.672%)
Validation starts
Test: [0/79]	Time 15.391 (15.39

In [51]:
print(model.layer1[0])

BasicBlock(
  (conv1): QuantConv2d(
    16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (weight_quant): weight_quantize_fn()
  )
  (conv2): QuantConv2d(
    16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (weight_quant): weight_quantize_fn()
  )
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)


In [109]:
# 2nd conv layer: Find integer inputs x_int and integer weight w_int 
# Similar to hw3 example

class SaveOutput:
    def __init__(self):
        self.outputs = []
    def __call__(self, module, module_in):
        self.outputs.append(module_in[0].detach().clone())
    def clear(self):
        self.outputs = []


save_output = SaveOutput()
conv_layer2 = model.layer1[0].conv1
conv2_in = conv_layer2.register_forward_pre_hook(save_output)

In [110]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Need to run the model after registering prehooks
model.eval()
# This stuff copied from HW3 solution
images, labels = next(iter(testloader))
with torch.no_grad():
    model(images.to(device))

# Clear outputs so that we don't have residual data
conv2_in.remove()

# This is the input to the 2nd conv layer
x_in = save_output.outputs[0]

# This is the layer weights: 
w_in = conv_layer2.weight

In [111]:
# Quantize the input and weights to integers

# Get alpha values
print("2nd conv layer params: ")
print(conv_layer2.show_params())

n_bits = conv_layer2.bit
print("# bits for x, w: " + str(n_bits))

alpha_x = conv_layer2.act_alpha
alpha_w = conv_layer2.weight_quant.wgt_alpha

def x_quantize(input, alpha, bits):
    in_div = input.div(alpha)
    in_clip = torch.clip(in_div, -1, 1)
    in_int = in_clip.mul(2 ** bits - 1).round()
    return in_int

def w_quantize(input, alpha, bits):
    # Divide by alpha, scale by 2 ** bits - 1, round to nearest int
    # weight_quantize_fn: subtracts mean, then divides by stdev
    mean = input.mean()
    stdev = input.std()
    in_norm = input.add(-mean).div(stdev)
    in_div = in_norm.div(alpha)
    in_clip = torch.clip(in_div, -1, 1)
    in_int = in_clip.mul(2 ** bits - 1).round()
    return in_int

x_int = x_quantize(x_in, alpha_x, n_bits)
w_int = w_quantize(w_in, alpha_w, n_bits-1) # Note w is signed so we -1 from bits


2nd conv layer params: 
clipping threshold weight alpha: 1.590000, activation alpha: 1.563000
None
# bits for x, w: 4


In [120]:
# Q1.5
# Partial sum from x_int and w_int
conv_layer_cp = torch.nn.Conv2d(16, 16, kernel_size=3, 
                           stride=1, padding=1, bias=False)

conv_layer_cp.weight = nn.Parameter(w_int)
out_int = conv_layer_cp(x_int)

# Rescale the integer psum with quantization step sizes
dx = alpha_x / (2 ** conv_layer2.bit - 1)
dw = alpha_w / (2 ** (conv_layer2.bit - 1) - 1)

quant_out = out_int.mul(dw).mul(dx)

print("quantized output:")
print(quant_out)

quantized output:
tensor([[[[-7.8115e-01, -1.3493e+00, -2.1541e+00,  ...,  2.1304e-01,
            1.1836e-01,  2.6038e-01],
          [ 7.3381e-01,  1.3256e+00,  2.9826e+00,  ...,  3.1009e+00,
            1.6570e-01, -5.6811e-01],
          [-5.9178e-01, -2.0831e+00, -4.2608e-01,  ..., -2.8169e+00,
           -5.3261e+00, -2.5328e+00],
          ...,
          [-4.6159e+00, -1.0747e+01, -1.1149e+01,  ..., -9.7763e+00,
           -1.2285e+01, -6.6516e+00],
          [-2.5802e+00, -1.0297e+01, -1.1433e+01,  ..., -9.5396e+00,
           -1.3871e+01, -9.3739e+00],
          [-7.5748e-01, -2.0121e+00, -3.9294e+00,  ..., -4.7106e+00,
           -5.2787e+00, -4.6869e+00]],

         [[-5.3261e+00, -6.6753e+00, -2.8879e+00,  ..., -3.2430e+00,
           -4.8763e+00,  3.5744e+00],
          [-7.0777e+00, -7.7169e+00, -1.2546e+00,  ..., -4.9710e-01,
            0.0000e+00,  1.0013e+01],
          [-5.5628e+00, -6.5096e+00,  3.5507e-01,  ...,  2.0831e+00,
            1.6097e+00,  1.0605e+01],
  

In [113]:
# Get the original output of the 2nd conv 
# Note: BN layer after conv
save_output2 = SaveOutput()
bn_layer2 = model.layer1[0].bn1
bn_in = bn_layer2.register_forward_pre_hook(save_output2)

In [114]:
# Evaluate the model again with the batch norm prehook attached
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Need to run the model after registering prehooks
model.eval()
# This stuff copied from HW3 solution
images, labels = next(iter(testloader))
with torch.no_grad():
    model(images.to(device))

# Clear outputs so that we don't have residual data
bn_in.remove()

# This is the input to the 2nd batch norm layer
x_out = save_output2.outputs[0]

In [115]:
print(x_out)

tensor([[[[-7.8115e-01, -1.3493e+00, -2.1541e+00,  ...,  2.1304e-01,
            1.1836e-01,  2.6038e-01],
          [ 7.3381e-01,  1.3256e+00,  2.9826e+00,  ...,  3.1009e+00,
            1.6570e-01, -5.6811e-01],
          [-5.9178e-01, -2.0831e+00, -4.2608e-01,  ..., -2.8169e+00,
           -5.3261e+00, -2.5328e+00],
          ...,
          [-4.6159e+00, -1.0747e+01, -1.1149e+01,  ..., -9.7763e+00,
           -1.2285e+01, -6.6516e+00],
          [-2.5802e+00, -1.0297e+01, -1.1433e+01,  ..., -9.5396e+00,
           -1.3871e+01, -9.3739e+00],
          [-7.5748e-01, -2.0121e+00, -3.9294e+00,  ..., -4.7106e+00,
           -5.2787e+00, -4.6869e+00]],

         [[-5.3261e+00, -6.6753e+00, -2.8879e+00,  ..., -3.2430e+00,
           -4.8763e+00,  3.5744e+00],
          [-7.0777e+00, -7.7169e+00, -1.2546e+00,  ..., -4.9710e-01,
            8.7971e-08,  1.0013e+01],
          [-5.5628e+00, -6.5096e+00,  3.5507e-01,  ...,  2.0831e+00,
            1.6097e+00,  1.0605e+01],
          ...,
     

In [121]:
# Difference between my quantized convolution and actual model output
F.mse_loss(x_out, quant_out)

tensor(3.5357e-12, device='cuda:0', grad_fn=<MseLossBackward0>)